In [1]:
import asyncio
import websockets
import json

BENIM_ID = "2300007951"
GIRILECEK_DERS = "Applied Deep Learning" 
SUREKLI_BAGLI_KALSIN = True 
KOPYA_CEKIYOR_MU = False

async def run_student():
    uri = "ws://localhost:8765"
    try:
        async with websockets.connect(uri) as websocket:
            print(f"🟢 [ÖĞRENCİ {BENIM_ID}] Sunucuya bağlanıldı!")

            await websocket.send(json.dumps({
                "action": "request_start_exam",
                "student_id": BENIM_ID,
                "exam_id": GIRILECEK_DERS
            }))
            
            response = json.loads(await websocket.recv())
            
            if response.get("status") == "error":
                print(f"❌ BAĞLANTI REDDEDİLDİ: {response.get('message')}")
                return 
            
            if response.get("status") == "success":
                token = response.get("session_token")
                
                # --- YENİ: SUNUCUDAN GELEN GÜVENLİK UYARISINI EKRANA BAS ---
                if response.get("warning"):
                    print(f"{response.get('warning')}")
                # -----------------------------------------------------------

                if response.get("reconnected") == True:
                    time_left = response.get("time_left_seconds")
                    print(f"<- 🔄 Bağlantı kurtarıldı! Sınava kalındığı yerden devam ediliyor: {time_left} saniye\n")
                else:
                    duration_mins = response.get("total_duration_minutes", 40)
                    time_left = duration_mins * 60
                    print(f"<- Sınav başladı! Alınan Süre: {duration_mins} dakika ({time_left} saniye)\n")
                
                dongu_limiti = 999999 if SUREKLI_BAGLI_KALSIN else 3
                
                for i in range(dongu_limiti):
                    await asyncio.sleep(2)
                    
                    if not KOPYA_CEKIYOR_MU:
                        time_left -= 2  
                        durum_mesaji = f"\r-> Güncelleme: Kalan süre {time_left}sn    "
                    else:
                        durum_mesaji = f"\r🚨 YASAKLI İŞLEM TESPİTİ! Yerel süre donduruldu: {time_left}sn    "

                    if time_left <= 0:
                        print("\nSüre bitti!")
                        break

                    await websocket.send(json.dumps({
                        "action": "status_update",
                        "student_id": BENIM_ID,
                        "session_token": token,
                        "timing": {"time_remaining_seconds": time_left},
                        "security": {"violation_alert": KOPYA_CEKIYOR_MU} 
                    }))
                    print(durum_mesaji, end="", flush=True)
                
                print(f"\n🔴 [ÖĞRENCİ {BENIM_ID}] Çıkış yapıldı (Koptu).")
    except Exception as e:
        print("❌ Hata:", e)

await run_student()

🟢 [ÖĞRENCİ 2300007951] Sunucuya bağlanıldı!
⚠️ DİKKAT: Eski sürüm (şifresiz) bir istemci kullanıyorsunuz!
<- Sınav başladı! Alınan Süre: 40 dakika (2400 saniye)

-> Güncelleme: Kalan süre 2302sn    ❌ Hata: sent 1011 (internal error) keepalive ping timeout; no close frame received
